In [6]:
# pip install torch torchvision torchaudio transformers scikit-learn pandas numpy

In [7]:
import pandas as pd
import numpy as np
import ast
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import LinearSVC

In [8]:
df = pd.read_csv('../court_cases.csv', parse_dates=['Date'])
df.head()

,Case_Number,Coram,Judge,Date,Tribunal_Court,Plaintiff_Name,Defendant_Name,Combined_Facts,Combined_Issue,Combined_Rule,Combined_Application,plaintiff_label,defendant_label
0,Suit 798/2007,Judith Prakash J,Judith Prakash,2009-07-06,High Court,ABB Holdings Pte Ltd,Sher Hock Guan Charles,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '19...","['Whether, based on the pre-2003 employment co...","['Under employment and company law, the obliga...",['ABB Holdings Pte Ltd issued and administered...,Claim Dismissed,Liable
1,Suit 798/2007,Judith Prakash J,Judith Prakash,2009-07-06,High Court,ABB Installation Materials (East Asia) Pte Ltd,Sher Hock Guan Charles,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '19...",['Whether the pre-2003 conduct of Sher Hock Gu...,['Company officers and senior managers must av...,['ABB Installation Materials (East Asia) Pte L...,Claim Dismissed,Liable
2,Suit 798/2007,Judith Prakash J,Judith Prakash,2009-07-06,High Court,ABB Industry Pte Ltd,Sher Hock Guan Charles,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '19...","['Whether, before and up to February 2003, the...",['Senior managers responsible for a business a...,['While serving as General Manager and Vice-Pr...,Claim Allowed,Liable
3,Suit No 477 of 2012; Suit No 1015 of 2012; Reg...,George Wei JC,George Wei,2014-02-14,High Court,Airtrust (Singapore) Pte Ltd (AT) – Suit No 47...,Kao Chai-Chau Linda,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '20...","['Whether, in Suit No 477 of 2012, losses to A...",['Where multiple persons are each legally resp...,"['Airtrust (Singapore) Pte Ltd, through the de...",Claim Allowed,Liable
4,Suit No 477 of 2012; Suit No 1015 of 2012; Reg...,George Wei JC,George Wei,2014-02-14,High Court,Airtrust (Singapore) Pte Ltd (AT) – Suit No 47...,Estate of Peter Fong,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '20...","['Whether, in Suit No 477 of 2012, losses to A...",['Where multiple persons are each legally resp...,"['Airtrust (Singapore) Pte Ltd, through the de...",Claim Allowed,Liable


In [9]:
df.columns

Index(['Case_Number', 'Coram', 'Judge', 'Date', 'Tribunal_Court',
       'Plaintiff_Name', 'Defendant_Name', 'Combined_Facts', 'Combined_Issue',
       'Combined_Rule', 'Combined_Application', 'plaintiff_label',
       'defendant_label'],
      dtype='str')

In [10]:
df = df.drop(columns=['Coram', 'Judge', 'Date', 'Plaintiff_Name', 'Defendant_Name', 'Combined_Rule', 'Combined_Application', 'plaintiff_label'])

In [11]:
df.head()

,Case_Number,Tribunal_Court,Combined_Facts,Combined_Issue,defendant_label
0,Suit 798/2007,High Court,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '19...","['Whether, based on the pre-2003 employment co...",Liable
1,Suit 798/2007,High Court,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '19...",['Whether the pre-2003 conduct of Sher Hock Gu...,Liable
2,Suit 798/2007,High Court,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '19...","['Whether, before and up to February 2003, the...",Liable
3,Suit No 477 of 2012; Suit No 1015 of 2012; Reg...,High Court,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '20...","['Whether, in Suit No 477 of 2012, losses to A...",Liable
4,Suit No 477 of 2012; Suit No 1015 of 2012; Reg...,High Court,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '20...","['Whether, in Suit No 477 of 2012, losses to A...",Liable


In [12]:
le = LabelEncoder()
df["label"] = le.fit_transform(df["defendant_label"])   # Liable=1, Not Liable=0
print("Classes:", le.classes_)

Classes: ['Liable' 'Not Liable']


In [13]:
df.head()

,Case_Number,Tribunal_Court,Combined_Facts,Combined_Issue,defendant_label,label
0,Suit 798/2007,High Court,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '19...","['Whether, based on the pre-2003 employment co...",Liable,0
1,Suit 798/2007,High Court,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '19...",['Whether the pre-2003 conduct of Sher Hock Gu...,Liable,0
2,Suit 798/2007,High Court,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '19...","['Whether, before and up to February 2003, the...",Liable,0
3,Suit No 477 of 2012; Suit No 1015 of 2012; Reg...,High Court,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '20...","['Whether, in Suit No 477 of 2012, losses to A...",Liable,0
4,Suit No 477 of 2012; Suit No 1015 of 2012; Reg...,High Court,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '20...","['Whether, in Suit No 477 of 2012, losses to A...",Liable,0


In [14]:
LEGALBERT_NAME   = "nlpaueb/legal-bert-base-uncased"
MAX_WORD_TOKENS  = 30          # Max tokens per sentence  (word encoder)
MAX_SENTENCES    = 20          # Max sentences per document (sentence encoder)
WORD_HIDDEN      = 64          # 64 forward + 64 backward = 128 dims per token
SENT_HIDDEN      = 64          # 64 forward + 64 backward = 128 dims per sentence
TRIBUNAL_DIM     = 16
DEVICE           = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. TEXT EXTRACTION HELPERS
# ─────────────────────────────────────────────────────────────────────────────

# Combined_Facts is a List of Lists of Dicts (LLD):  k x n x {128+16+16}
# Each dict has keys: Fact_Type, Fact_Date, + text fields
# We extract per-fact:  (sentence_text, fact_type_str, fact_date_str)

def parse_if_string(obj):
    if isinstance(obj, str):
        try:
            return ast.literal_eval(obj)
        except Exception:
            return obj
    return obj

def clean(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def extract_facts(raw):
    """
    Returns a list of dicts: [{text, fact_type, fact_date}, ...]
    One entry per fact sentence.  Outer list = facts for one case.
    """
    raw = parse_if_string(raw)
    records = []
    # raw may be [[dict, ...], ...] or [dict, ...] — flatten one level
    if isinstance(raw, list) and raw and isinstance(raw[0], list):
        flat = [item for sublist in raw for item in sublist]
    else:
        flat = raw if isinstance(raw, list) else [raw]

    for item in flat:
        item = parse_if_string(item)
        if not isinstance(item, dict):
            continue
        fact_type = clean(item.get("Fact_Type", "unknown"))
        fact_date = clean(item.get("Fact_Date", "unknown"))
        # Collect all remaining string values as sentence text
        text_parts = [
            clean(v) for k, v in item.items()
            if k not in ("Fact_Type", "Fact_Date") and isinstance(v, str)
        ]
        text = " ".join(text_parts).strip()
        if text:
            records.append({"text": text, "fact_type": fact_type, "fact_date": fact_date})
    return records

def extract_issue(raw):
    """Returns a list of sentence strings from Combined_Issue (LoS)."""
    raw = parse_if_string(raw)
    if isinstance(raw, list):
        return [clean(s) for s in raw if str(s).strip()]
    return [clean(str(raw))]

In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. LEGALBERT TOKENIZER + MODEL  (frozen — used as embedding layer)
# ─────────────────────────────────────────────────────────────────────────────

tokenizer   = AutoTokenizer.from_pretrained(LEGALBERT_NAME)
legalbert   = AutoModel.from_pretrained(LEGALBERT_NAME).to(DEVICE)

for param in legalbert.parameters():
    param.requires_grad = False

legalbert.eval()

LEGALBERT_DIM = legalbert.config.hidden_size  # 768

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20228.45it/s]
BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. LEGALBERT SENTENCE TOKENIZER
#    Returns token embeddings (max_word_tokens, 768) for one sentence string
# ─────────────────────────────────────────────────────────────────────────────

def embed_sentence_tokens(sentence_text, max_len=MAX_WORD_TOKENS):
    """
    Returns (max_len, 768) token-level embeddings from LegalBERT (frozen).
    Pads or truncates to max_len.
    """
    enc = tokenizer(
        sentence_text,
        max_length=max_len,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    input_ids      = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)

    with torch.no_grad():
        out = legalbert(input_ids=input_ids, attention_mask=attention_mask)

    # out.last_hidden_state: (1, max_len, 768)
    return out.last_hidden_state.squeeze(0)   # (max_len, 768)

In [18]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. WORD ENCODER  —  2x Bi-GRU  (64 hidden each dir → 128 per token)
#    Fact_Type & Fact_Date embeddings are concatenated ONLY for Combined_Facts
# ─────────────────────────────────────────────────────────────────────────────

# Fact_Type / Fact_Date are short strings — we embed them with LegalBERT CLS
# then project to a small dim so the concat doesn't dominate
FACT_META_DIM = 16

fact_meta_proj = nn.Linear(LEGALBERT_DIM, FACT_META_DIM).to(DEVICE)

word_bigru = nn.GRU(
    input_size    = LEGALBERT_DIM + FACT_META_DIM + FACT_META_DIM,  # 768+16+16
    hidden_size   = WORD_HIDDEN,   # 64  → bidirectional gives 128
    num_layers    = 2,
    batch_first   = True,
    bidirectional = True,
    dropout       = 0.3
).to(DEVICE)

word_bigru_plain = nn.GRU(
    input_size    = LEGALBERT_DIM,   # no meta — for Combined_Issue
    hidden_size   = WORD_HIDDEN,
    num_layers    = 2,
    batch_first   = True,
    bidirectional = True,
    dropout       = 0.3
).to(DEVICE)


def get_cls_vector(text):
    """Returns (LEGALBERT_DIM,) CLS embedding for a short text string."""
    enc = tokenizer(text, return_tensors="pt",
                    max_length=16, padding="max_length",
                    truncation=True).to(DEVICE)
    with torch.no_grad():
        out = legalbert(**enc)
    return out.last_hidden_state[:, 0, :].squeeze(0)   # (768,)

In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# 6. WORD ATTENTION
#    MLP + add + norm  →  context vector u_w
# ─────────────────────────────────────────────────────────────────────────────

word_attn_mlp  = nn.Linear(WORD_HIDDEN * 2, WORD_HIDDEN * 2).to(DEVICE)
word_attn_norm = nn.LayerNorm(WORD_HIDDEN * 2).to(DEVICE)
word_context   = nn.Parameter(torch.randn(WORD_HIDDEN * 2).to(DEVICE))


def word_attention(h):
    """
    h      : (seq_len, 128)
    returns: (128,)  weighted sentence vector
    """
    # MLP + residual + LayerNorm
    u     = word_attn_norm(h + torch.tanh(word_attn_mlp(h)))       # (seq_len, 128)
    score = torch.matmul(u, word_context)                          # (seq_len,)
    alpha = torch.softmax(score, dim=0)                            # (seq_len,)
    s     = torch.matmul(alpha.unsqueeze(0), h).squeeze(0)         # (128,)
    return s


In [20]:
# ─────────────────────────────────────────────────────────────────────────────
# 7. ENCODE ONE SENTENCE  →  128-dim vector
#    with_meta=True  for Combined_Facts (appends Fact_Type + Fact_Date)
# ─────────────────────────────────────────────────────────────────────────────

def encode_sentence(sentence_text,
                    fact_type=None, fact_date=None,
                    with_meta=False):
    """
    Returns (128,) sentence-level vector via Word Encoder + Word Attention.
    """
    tok_emb = embed_sentence_tokens(sentence_text)   # (max_word_tokens, 768)

    if with_meta and fact_type is not None and fact_date is not None:
        ft_vec  = fact_meta_proj(get_cls_vector(fact_type))    # (16,)
        fd_vec  = fact_meta_proj(get_cls_vector(fact_date))    # (16,)
        # Broadcast and concat along last dim → (max_word_tokens, 768+16+16)
        ft_exp  = ft_vec.unsqueeze(0).expand(tok_emb.size(0), -1)
        fd_exp  = fd_vec.unsqueeze(0).expand(tok_emb.size(0), -1)
        inp     = torch.cat([tok_emb, ft_exp, fd_exp], dim=-1)  # (T, 800)
        h, _    = word_bigru(inp.unsqueeze(0))                  # (1, T, 128)
    else:
        h, _    = word_bigru_plain(tok_emb.unsqueeze(0))        # (1, T, 128)

    h = h.squeeze(0)                                            # (T, 128)
    return word_attention(h)                                    # (128,)


In [21]:
# ─────────────────────────────────────────────────────────────────────────────
# 8. SENTENCE ENCODER  —  1-2x Bi-GRU  (64 hidden → 128 per sentence)
#    + Sentence Attention  (MLP + add + norm + context vector u_s)
# ─────────────────────────────────────────────────────────────────────────────

sent_bigru = nn.GRU(
    input_size    = WORD_HIDDEN * 2,   # 128
    hidden_size   = SENT_HIDDEN,       # 64 → bidirectional = 128
    num_layers    = 2,
    batch_first   = True,
    bidirectional = True,
    dropout       = 0.3
).to(DEVICE)

sent_attn_mlp  = nn.Linear(SENT_HIDDEN * 2, SENT_HIDDEN * 2).to(DEVICE)
sent_attn_norm = nn.LayerNorm(SENT_HIDDEN * 2).to(DEVICE)
sent_context   = nn.Parameter(torch.randn(SENT_HIDDEN * 2).to(DEVICE))


def sentence_attention(h):
    """
    h      : (num_sentences, 128)
    returns: (128,)  document vector
    """
    u     = sent_attn_norm(h + torch.tanh(sent_attn_mlp(h)))
    score = torch.matmul(u, sent_context)
    alpha = torch.softmax(score, dim=0)
    doc   = torch.matmul(alpha.unsqueeze(0), h).squeeze(0)
    return doc


def encode_document(sentence_vectors):
    """
    sentence_vectors : list of (128,) tensors
    Returns          : (128,) document vector
    Pads / truncates to MAX_SENTENCES.
    """
    if len(sentence_vectors) == 0:
        return torch.zeros(SENT_HIDDEN * 2).to(DEVICE)

    # Pad or truncate
    vecs = sentence_vectors[:MAX_SENTENCES]
    while len(vecs) < MAX_SENTENCES:
        vecs.append(torch.zeros(WORD_HIDDEN * 2).to(DEVICE))

    mat  = torch.stack(vecs, dim=0).unsqueeze(0)   # (1, MAX_SENTENCES, 128)
    h, _ = sent_bigru(mat)                          # (1, MAX_SENTENCES, 128)
    h    = h.squeeze(0)                             # (MAX_SENTENCES, 128)
    return sentence_attention(h)                    # (128,)

In [22]:
# ─────────────────────────────────────────────────────────────────────────────
# 9. TRIBUNAL_COURT  →  16-dim one-hot vector
# ─────────────────────────────────────────────────────────────────────────────

tribunal_enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
tribunal_enc.fit(df[["Tribunal_Court"]])

# Pad / trim to exactly TRIBUNAL_DIM
def encode_tribunal(court_val):
    vec = tribunal_enc.transform([[court_val]])[0]
    if len(vec) < TRIBUNAL_DIM:
        vec = np.pad(vec, (0, TRIBUNAL_DIM - len(vec)))
    else:
        vec = vec[:TRIBUNAL_DIM]
    return torch.tensor(vec, dtype=torch.float32).to(DEVICE)

In [23]:
# ─────────────────────────────────────────────────────────────────────────────
# 10. BUILD FEATURE MATRIX  (one row per case)
#     Late Fusion: concat [facts_doc(128) | issue_doc(128) | tribunal(16)]
#     → 272-dim feature vector per case
# ─────────────────────────────────────────────────────────────────────────────

def build_feature_vector(row):
    # ── Combined_Facts ──────────────────────────────────────────────────────
    facts = extract_facts(row["Combined_Facts"])
    facts_sent_vecs = [
        encode_sentence(f["text"], f["fact_type"], f["fact_date"], with_meta=True)
        for f in facts if f["text"]
    ]
    facts_doc_vec = encode_document(facts_sent_vecs)       # (128,)

    # ── Combined_Issue ───────────────────────────────────────────────────────
    issues = extract_issue(row["Combined_Issue"])
    issue_sent_vecs = [
        encode_sentence(s, with_meta=False)
        for s in issues if s
    ]
    issue_doc_vec = encode_document(issue_sent_vecs)       # (128,)

    # ── Tribunal_Court ───────────────────────────────────────────────────────
    tribunal_vec = encode_tribunal(row["Tribunal_Court"])  # (16,)

    # ── Late Fusion ──────────────────────────────────────────────────────────
    fused = torch.cat([facts_doc_vec, issue_doc_vec, tribunal_vec], dim=0)  # (272,)
    return fused.detach().cpu().numpy()


# Extract case IDs for group-based splitting (prevents data leakage)
case_ids = df['Case_Number'].values

print("Building feature matrix — this may take a few minutes...")

X_all = np.vstack([build_feature_vector(row) for _, row in df.iterrows()])
print(f"Feature matrix shape: {X_all.shape}")   # (n_cases, 272)

Building feature matrix — this may take a few minutes...


c:\Users\tanke\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\tanke\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\tanke\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\tanke\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\tanke\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserW

Feature matrix shape: (424, 272)


c:\Users\tanke\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 11. LABELS
# ─────────────────────────────────────────────────────────────────────────────

# Manual encoding: 0 = Not Liable, 1 = Liable
label_map = {'Not Liable': 0, 'Liable': 1}
y_all = df["defendant_label"].map(label_map).values

print("Label mapping:")
print(f"  0 = Not Liable")
print(f"  1 = Liable")
print(f"\nEncoded labels - value counts:")
unique, counts = np.unique(y_all, return_counts=True)
for label_val, count in zip(unique, counts):
    label_name = {0: 'Not Liable', 1: 'Liable'}[label_val]
    print(f"  {label_val} ({label_name}): {count}")


Classes: ['Liable' 'Not Liable']


In [25]:
# ─────────────────────────────────────────────────────────────────────────────
# 12. TRAIN / TEST SPLIT (GROUP-BASED TO PREVENT DATA LEAKAGE)
#     Uses GroupShuffleSplit to ensure all rows of the same case go together
# ─────────────────────────────────────────────────────────────────────────────

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X_all, y_all, groups=case_ids))

X_train, X_test = X_all[train_idx], X_all[test_idx]
y_train, y_test = y_all[train_idx], y_all[test_idx]

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Unique train cases: {len(set(case_ids[train_idx]))}  |  Unique test cases: {len(set(case_ids[test_idx]))}")

Train: (347, 272)  |  Test: (77, 272)
Unique train cases: 109  |  Unique test cases: 28


In [26]:
# ─────────────────────────────────────────────────────────────────────────────
# 13. LINEAR SVM
#     Wrapped in CalibratedClassifierCV so we get probability estimates too
# ─────────────────────────────────────────────────────────────────────────────

base_svm = LinearSVC(
    C=1.0,
    max_iter=5000,
    class_weight="balanced",   # handles class imbalance
    random_state=42
)

svm_model = CalibratedClassifierCV(base_svm, cv=5)
svm_model.fit(X_train, y_train)

print("SVM training complete.")

SVM training complete.


In [27]:
# ─────────────────────────────────────────────────────────────────────────────
# 14. EVALUATION
# ─────────────────────────────────────────────────────────────────────────────

y_pred  = svm_model.predict(X_test)
y_proba = svm_model.predict_proba(X_test)

print(classification_report(y_test, y_pred, target_names=le.classes_))

# Per-sample probability output
proba_df = pd.DataFrame(y_proba, columns=[f"P({c})" for c in le.classes_])
proba_df["Predicted"]  = le.inverse_transform(y_pred)
proba_df["True_Label"] = le.inverse_transform(y_test)
print(proba_df.head(10))

              precision    recall  f1-score   support

      Liable       0.44      0.85      0.58        34
  Not Liable       0.55      0.14      0.22        43

    accuracy                           0.45        77
   macro avg       0.49      0.50      0.40        77
weighted avg       0.50      0.45      0.38        77

   P(Liable)  P(Not Liable) Predicted  True_Label
0   0.519828       0.480172    Liable      Liable
1   0.515590       0.484410    Liable      Liable
2   0.517346       0.482654    Liable      Liable
3   0.510987       0.489013    Liable      Liable
4   0.507019       0.492981    Liable  Not Liable
5   0.509377       0.490623    Liable  Not Liable
6   0.508014       0.491986    Liable      Liable
7   0.505183       0.494817    Liable      Liable
8   0.518012       0.481988    Liable  Not Liable
9   0.501376       0.498624    Liable  Not Liable


In [28]:
df['label'].value_counts()

label
1    213
0    211
Name: count, dtype: int64